# OA-ReactDiff candidate-generation pilot

Open this notebook from GitHub and use a GPU runtime. It keeps the historical OA environment separate from HORM and writes every completed reaction to Drive, so an interrupted runtime can resume safely.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPOSITORY_URL = 'https://github.com/chenruduan/OAReactDiff.git'
REPOSITORY_REF = 'agent/oa-failure-audit'
REPO = Path('/content/OAReactDiff')
OUTPUT_ROOT = Path('/content/drive/MyDrive/OAReactDiff/audit_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
if not (REPO / '.git').is_dir():
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --branch {REPOSITORY_REF} {REPOSITORY_URL} {REPO}
else:
    !git -C {REPO} fetch --depth 1 origin {REPOSITORY_REF}
    !git -C {REPO} checkout --detach FETCH_HEAD
!git -C {REPO} rev-parse HEAD
assert (REPO / 'experiments/oa_failure_audit/generate_candidates.py').is_file()

In [ ]:
import hashlib

CHECKPOINT = REPO / 'pretrained-ts1x-diff.ckpt'
CHECKPOINT_URL = 'https://media.githubusercontent.com/media/chenruduan/OAReactDiff/main/pretrained-ts1x-diff.ckpt'
CHECKPOINT_SHA256 = 'b02106f36a8033118e3de3c7bc3ba1d769f677e4d47c714a3eea20e6b6c787cb'

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not CHECKPOINT.is_file() or sha256(CHECKPOINT) != CHECKPOINT_SHA256:
    !curl -L --fail --retry 3 --output {CHECKPOINT} {CHECKPOINT_URL}
assert sha256(CHECKPOINT) == CHECKPOINT_SHA256
!cd {REPO} && bash experiments/oa_failure_audit/setup_colab_generation.sh

In [ ]:
MAMBA = '/usr/local/bin/micromamba'
ENV = 'oa-generation'
!cd {REPO} && {MAMBA} run -n {ENV} python experiments/oa_failure_audit/generate_candidates.py --dry-run --indices 0 --samples-per-reaction 1

## Timed 8 × 8 pilot

The first run uses the lightweight `2/2` repaint setting. Re-running the cell resumes at the first incomplete reaction.

In [ ]:
PILOT_OUTPUT = OUTPUT_ROOT / 'generation_8x8_r2_j2'
!cd {REPO} && {MAMBA} run -n {ENV} python experiments/oa_failure_audit/generate_candidates.py \
    --device cuda --max-reactions 8 --samples-per-reaction 8 \
    --resamplings 2 --jump-length 2 --output-dir {PILOT_OUTPUT} --resume

In [ ]:
import json
timing = json.loads((PILOT_OUTPUT / 'timing_summary.json').read_text())
seconds_per_candidate = timing['mean_generation_seconds_per_candidate']
print(json.dumps(timing, indent=2))
print(f'Projected 50 x 40: {seconds_per_candidate * 2000 / 3600:.2f} GPU-hours')
print(f'Projected 1073 x 40: {seconds_per_candidate * 42920 / 3600:.2f} GPU-hours')

## Paper-style timing control

Run this only after the lightweight pilot succeeds. It uses the repository evaluation setting `10/10` and a separate output directory.

In [ ]:
RUN_PAPER_STYLE = False
if RUN_PAPER_STYLE:
    paper_output = OUTPUT_ROOT / 'generation_8x8_r10_j10'
    !cd {REPO} && {MAMBA} run -n {ENV} python experiments/oa_failure_audit/generate_candidates.py \
        --device cuda --max-reactions 8 --samples-per-reaction 8 \
        --resamplings 10 --jump-length 10 --output-dir {paper_output} --resume